In [1]:
import os
import shutil
from tqdm import tqdm

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9

# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# api.py
COPY api.py .

# app.py
COPY app.py .

# cls_parser.pkl
COPY cls_parser.pkl .

# datascience_rsa_key.p8
COPY datascience_rsa_key.p8 .

# functions_app
COPY functions_app.py .

# preprocessing.py
COPY preprocessing.py .

# df_aa.csv
COPY df_aa.csv .

# static
COPY static ./static

# templates
COPY templates ./templates

# run script when image is run
CMD ["python3", "app.py"]

Writing Dockerfile


### Build and push to ECR

In [3]:
%%sh

# name the image
image=gen13-payload-comparison

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 638B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.9
#2 DONE 0.6s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [ 1/14] FROM docker.io/library/python:3.9@sha256:c17c71e1f5f258803a6b7c391f8013adbf84285af54c2a811de4a5a1ac5a8676
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 14.00kB done
#5 DONE 0.0s

#6 [ 2/14] RUN apt-get update
#6 CACHED

#7 [ 3/14] RUN pip install --upgrade pip
#7 CACHED

#8 [ 5/14] RUN pip install -r requirements.txt
#8 CACHED

#9 [ 4/14] COPY requirements.txt .
#9 CACHED

#10 [ 6/14] COPY api.py .
#10 CACHED

#11 [ 7/14] COPY app.py .
#11 DONE 0.0s

#12 [ 8/14] COPY cls_parser.pkl .
#12 DONE 0.0s

#13 [ 9/14] COPY datascience_rsa_key.p8 .
#13 DONE 0.1s

#14 [10/14] COPY functions_app.py .
#14 DONE 0.0s

#15 [11/14] COPY preprocessing.py .

Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'gen13-payload-comparison' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/gen13-payload-comparison]
37e720244c2c: Preparing
47fa11627661: Preparing
5f165b4a48a2: Preparing
2e4b653cd512: Preparing
868b58ee4236: Preparing
cc866ce25c61: Preparing
82923c1bc91b: Preparing
6092c7916419: Preparing
99691934e35e: Preparing
c63eeb79f24d: Preparing
46aa9ef82a5b: Preparing
4fb16df75875: Preparing
ad78efb2b02b: Preparing
60a159600b22: Preparing
ee959616fc20: Preparing
d0e85779261a: Preparing
dafb8aed9f7f: Preparing
41d4dc7516bb: Preparing
c0f51bbdc37d: Preparing
91b542912d12: Preparing
46aa9ef82a5b: Waiting
4fb16df75875: Waiting
ad78efb2b02b: Waiting
60a159600b22: Waiting
ee959616fc20: Waiting
d0e85779261a: Waiting
dafb8aed9f7f: Waiting
41d4dc7516bb: Waiting
c0f51bbdc37d: Waiting
91b542912d12: Waiting
cc866ce25c61: Waiting
82923c1bc91b: Waiting
6092c7916419: Waiting
99691934e35e: Waiting
c63eeb79f24d: Waiting
47fa11627661: Layer already exists
37e720244c2c: Layer already exists
2e4b653cd512: Laye

### Clean up Dockerfile

In [4]:
try:
    os.remove('Dockerfile')
except FileNotFoundError:
    pass